In [ ]:
# 버전 확인 및 임포트
import chonkie, transformers, tokenizers, sentence_transformers
print("chonkie:", chonkie.__version__)
print("transformers:", transformers.__version__)
print("tokenizers:", tokenizers.__version__)
print("sentence_transformers:", sentence_transformers.__version__)

c:\Users\User\anaconda3\envs\chunk\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


chonkie: 1.3.1
transformers: 4.46.3
tokenizers: 0.20.3
sentence_transformers: 3.4.1


In [17]:
import json
from chonkie import SemanticChunker, SentenceTransformerEmbeddings

MODEL = "BAAI/bge-m3"        # huggingface에서 모델+토크나이저 전체 다운로드 (약 2GB)
INPUT_PATH = "C:\\Users\\User\\Desktop\\민원처리법_hwpx변환.md"         # 처리할 실제 md 파일 경로
OUTPUT_PATH = "C:\\Users\\User\\Desktop\\chunks.jsonl"    

THRESHOLD = 0.3    # 의미 경계 민감도(0~1). 높을수록 덜 자름(큰 청크), 낮을수록 잘게.
CHUNK_SIZE = 1024  # 청크 최대 토큰 수 (이 한도 넘으면 강제로 끊음)
MIN_SENTENCES = 2  # 한 청크가 최소한 가져야 하는 문장 개수

## 임베딩 모델 로드 (모델+토크나이저 전체 다운로드, 약 2GB, huggingface 계정 불필요)

In [18]:
embeddings = SentenceTransformerEmbeddings(MODEL)
print("BGE-M3 로드 완료, 임베딩 차원:", embeddings.dimension)

BGE-M3 로드 완료, 임베딩 차원: 1024


## 청커 생성 + 청킹

In [19]:
chunker = SemanticChunker(
    embedding_model=embeddings,
    threshold=THRESHOLD,
    chunk_size=CHUNK_SIZE,
    min_sentences=MIN_SENTENCES,
)

with open(INPUT_PATH, encoding="utf-8") as f:
    text = f.read()

chunks = chunker(text)
print(f"→ {len(chunks)}개 청크 생성")

→ 39개 청크 생성


## 미리보기

In [20]:
for i, c in enumerate(chunks[:5]):
    preview = c.text.strip().replace("\n", " ")[:60]
    print(f"[{i}] (토큰≈{c.token_count}) {preview}")

[0] (토큰≈111) # 민원 처리에 관한 법률 (약칭: 민원처리법)  [시행 2022. 7. 12.] [법률 제18748호, 2
[1] (토큰≈61) ---  ## 제1장 총칙  **제1조(목적)** 이 법은 민원 처리에 관한 기본적인 사항을 규정하여 민원의
[2] (토큰≈400) **제2조(정의)** 이 법에서 사용하는 용어의 뜻은 다음과 같다.  1. "민원"이란 민원인이 행정기관에 
[3] (토큰≈210) 3. "행정기관"이란 다음 각 목의 자를 말한다.     가. 국회ㆍ법원ㆍ헌법재판소ㆍ중앙선거관리위원회의 행정
[4] (토큰≈133) 4. "처분"이란 「행정절차법」 제2조제2호의 처분을 말한다.  5. "복합민원"이란 하나의 민원 목적을 실


In [11]:
## JSONL 저장

In [12]:
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    for i, c in enumerate(chunks):
        record = {"chunk_id": i, "text": c.text.strip(), "token_count": c.token_count}
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"저장 완료: {OUTPUT_PATH}")

저장 완료: C:\Users\User\Desktop\chunks.jsonl
